In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data.sampler import SubsetRandomSampler
import torchvision.transforms as transforms
from torch.utils.data import random_split
import matplotlib.pyplot as plt
from torchmetrics.classification import F1Score, Precision, Recall

ModuleNotFoundError: No module named 'torch'

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

np.random.seed(50)
dataset = torchvision.datasets.ImageFolder(root="???", transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, num_workers=1)
valid_loader = torch.utils.data.DataLoader(val_set, batch_size=64, num_workers=1)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=64, num_workers=1)

#work on this later

In [ ]:
class Baseline(nn.Module):
    def __init__(self):
        super(Baseline, self).__init__()
        self.layer1 = nn.Linear(74, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, 64)
        self.layer4 = nn.Linear(64, 5)

    def forward(self, img):
        flattened = img.view(-1, 74)
        activation1 = self.layer1(flattened)
        activation1 = F.ReLU(activation1)
        activation2 = self.layer2(activation1)
        activation2 = F.ReLU(activation2)
        activation3 = self.layer3(activation2)
        activation3 = F.ReLU(activation3)
        activation4 = self.layer4(activation3)
        return activation4

baselineModelOne = Baseline()

In [ ]:
def getF1Score(pred, target):
    f1 = F1Score(task="multiclass", num_classes=)
    score = f1(pred, target)

    return score 

def getPrecision(pred, target):
    precision = Precision(task="multiclass", num_classes=3)
    score = precision(pred, target)

    return score

def getRecall(pred, target):
    recall = Recall(task="multiclass", num_classes=3)
    score = recall(pred, target)

    return score

def train(model, train_loader, valid_loader, num_epochs=50, learning_rate=1e-4):
    """Training loop."""
    torch.manual_seed(42)
    criterion = nn.CrossEntropyLoss
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = {"trainLoss": [], "validLoss": [], "trainF1Score": [], "validF1Score": [], \
                "trainPrecision": [], "validPrecision": [], "trainRecall": [], "validRecall": []
    }

    for epoch in range(num_epochs):
        model.train()
        trainLoss = 0.0
        for input, label in train_loader:
            output = model(input)
            optimizer.zero_grad()
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()
            #Gets the scalar loss and multiplies it by the batch s to get the batch loss
            trainLoss += loss.item() * data.size(0)
        #converts total sum back into average
        trainLoss /= len(train_loader.dataset)

        model.eval()
        validLoss = 0.0
        with torch.no_grad():
            for input, label in valid_loader:
                output = model(input)
                loss = criterion(output, label)
                validLoss += loss.item() * data.size(0) #fix this
        validLoss /= len(valid_loader.dataset)

        #f1
        trainF1Score = getF1Score(model, train_loader)
        validF1Score = getF1Score(model, valid_loader)

        #precision
        trainPrecision = getPrecision(model, train_loader)
        validPrecision = getPrecision(model, valid_loader)

        #recall
        trainRecall = getRecall(model, train_loader)
        validRecall = getRecall(model, valid_loader)

        history["train_loss"].append(train_loss)
        history["valid_loss"].append(valid_loss)
        history["trainF1Score"].append(trainF1Score)
        history["validF1Score"].append(validF1Score)
        history["trainPrecision"].append(trainPrecision)
        history["validPrecision"].append(validPrecision)
        history["trainRecall"].append(trainRecall)
        history["validRecall"].append(validRecall)


        print(f"Epoch [{epoch+1}/{num_epochs}]  "
              f"Train Loss: {trainLoss:.6f}  "
              f"Valid Loss: {validLoss:.6f}  "
              f"Train F1 Score: {trainF1Score:.4f}  "
              f"Valid F1 Score: {validF1Score:.4f}"
              f"Train Precision: {trainPrecision:.4f}  "
              f"Valid Precision: {validPrecision:.4f}"
              f"Train Recall: {trainRecall:.4f}  "
              f"Valid Recall: {validRecall:.4f}" 
        )
    return model, history